# LSTM + Greedy [입력 언어 인식 및 6방향 통합 번역]

**하나의 LSTM Seq2Seq 모델**이 숫자(`num`), 영어(`en`), 한국어(`ko`)의 입력 언어를 인식하고 나머지 두 언어로 번역합니다.

- 공통 **Encoder LSTM**의 최종 hidden state → `lang_fc`로 입력 언어를 분류
- 같은 Encoder 상태 → **Decoder LSTM**으로 전달해 목표 언어별 번역 생성
- 목표 언어는 Decoder 시작 토큰 `<2num>`, `<2en>`, `<2ko>`로 지정
- 학습 손실 = 번역 CrossEntropy + `LANG_LOSS_WEIGHT` × 언어 분류 CrossEntropy

## 1. 설정

- 필요 패키지: `torch`, `tqdm`.
- PyTorch 설치는 운영체제와 CUDA 환경에 맞춰 [공식 안내](https://pytorch.org/get-started/locally/)를 이용하세요.
- `num2txt.txt`, `num2kor.txt`가 노트북 실행 폴더에 있어야 합니다.

In [ ]:
import random, time, re
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from tqdm import tqdm

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, EMB_DIM, HIDDEN_DIM, EPOCHS = 128, 128, 256, 30
LANG_LOSS_WEIGHT = 0.3  # 튜닝 전 초기 설정
MODEL_FILE = Path('lstm_greedy_multitask.pt')
TRAIN = True  # 최초 실행: True / 저장된 모델만 불러올 때: False
LANGS = ('num', 'en', 'ko')
LANG_TO_ID = {lang: i for i, lang in enumerate(LANGS)}
LANGS_LABEL = {'num': '숫자', 'en': '영어', 'ko': '한국어'}
print('Device:', DEVICE)

Device: cuda


## 2. 데이터 읽기, 숫자 단위 분할, 문자 어휘 만들기

영어·한국어 파일의 **숫자 키**를 기준으로 합칩니다. 같은 숫자의 표현이 학습과 테스트에 걸쳐 섞이지 않도록 **숫자 레코드를 먼저 분할**한 후 6방향 쌍을 생성합니다.

언어 판별에 정답 레이블을 제공하는 것은 **학습 시에만** 해당합니다. 추론에서는 `src` 레이블을 입력하지 않습니다.

In [ ]:
def load_file(path):
    with open(path, encoding='utf-8') as f:
        return dict(line.rstrip('\n').split('\t', 1)
                    for line in f if line.strip())

# 학습/추론 모두 동일한 전처리를 적용하며, 언어 판별 규칙은 넣지 않음
def normalize_text(text):
    return ' '.join(str(text).strip().lower().replace(',', '').split())

# 입력 언어와 무관한 문자 단위 토큰화
def tokenize(text):
    return list(normalize_text(text))

en = load_file('num2txt.txt')
ko = load_file('num2kor.txt')
records = {}
for num, english in en.items():
    key = normalize_text(num)
    records.setdefault(key, {'num': key})['en'] = normalize_text(english)
for num, korean in ko.items():
    key = normalize_text(num)
    records.setdefault(key, {'num': key})['ko'] = normalize_text(korean)
rows = list(records.values())

train_rows, val_rows, test_rows = random_split(
    rows, [0.8, 0.1, 0.1], generator=torch.Generator().manual_seed(SEED)
)

def make_pairs(data):
    pairs = []
    for texts in data:
        for src in LANGS:
            for tgt in LANGS:
                if src != tgt and src in texts and tgt in texts:
                    pairs.append((src, tgt, texts[src], texts[tgt]))
    return pairs

train_pairs = make_pairs(train_rows)
val_pairs = make_pairs(val_rows)
test_pairs = make_pairs(test_rows)

special = ['<pad>', '<bos>', '<eos>', '<unk>'] + [f'<2{x}>' for x in LANGS]
tokens = set()
for src, tgt, a, b in train_pairs:
    tokens.update(tokenize(a))
    tokens.update(tokenize(b))
vocab = special + sorted(tokens - set(special))
stoi = {token: i for i, token in enumerate(vocab)}  # 모델에 넣기 위한 변환
itos = {i: token for token, i in stoi.items()}  # 모델의 출력을 읽기 위한 변환
PAD, BOS, EOS, UNK = (stoi[x] for x in special[:4])

print('English records:', len(en))
print('Korean records:', len(ko))
print('Unique numbers:', len(rows))
print('Vocabulary:', len(vocab))
print('Training pairs:', len(train_pairs))
print('Validation pairs:', len(val_pairs))
print('Test pairs:', len(test_pairs))

English records: 100000
Korean records: 100000
Unique numbers: 187262
Vocabulary: 54
Training pairs: 340204
Validation pairs: 42780
Test pairs: 42492


## 3. Dataset: 원문, 번역 정답, 입력 언어 레이블

- `x`: 원문 문자의 ID + `<eos>`
- `y`: `<2목표언어>` + 목표 문자의 ID + `<eos>`
- `lang`: `num/en/ko`의 분류 정답(0/1/2)

In [3]:
def ids(text):
    return [stoi.get(token, UNK) for token in tokenize(text)]

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        src, tgt, a, b = self.pairs[index]
        x = ids(a) + [EOS]
        y = [stoi[f'<2{tgt}>']] + ids(b) + [EOS]
        lang = LANG_TO_ID[src]
        return (torch.tensor(x, dtype=torch.long),
                torch.tensor(y, dtype=torch.long),
                torch.tensor(lang, dtype=torch.long))

def collate(batch):
    x, y, lang = zip(*batch)
    return (pad_sequence(x, batch_first=True, padding_value=PAD),
            pad_sequence(y, batch_first=True, padding_value=PAD),
            torch.stack(lang))

train_loader = DataLoader(TranslationDataset(train_pairs), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate)
val_loader = DataLoader(TranslationDataset(val_pairs), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate)
print('Batch format: (input, target, source_language)')

Batch format: (input, target, source_language)


## 4. LSTM Seq2Seq + 입력 언어 분류 헤드

In [4]:
class Seq2Seq(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMB_DIM, padding_idx=PAD)
        self.encoder = nn.LSTM(EMB_DIM, HIDDEN_DIM, batch_first=True)
        self.decoder = nn.LSTM(EMB_DIM, HIDDEN_DIM, batch_first=True)
        self.fc = nn.Linear(HIDDEN_DIM, vocab_size)
        self.lang_fc = nn.Linear(HIDDEN_DIM, len(LANGS))

    def encode(self, x):
        lengths = (x != PAD).sum(1).cpu()
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths, batch_first=True,
                                      enforce_sorted=False)
        _, state = self.encoder(packed)
        hidden, cell = state
        lang_logits = self.lang_fc(hidden[-1])
        return state, lang_logits

    def forward(self, x, y):
        state, lang_logits = self.encode(x)
        out, _ = self.decoder(self.embedding(y[:, :-1]), state)
        translation_logits = self.fc(out)
        return translation_logits, lang_logits

model = Seq2Seq(len(vocab)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
translation_criterion = nn.CrossEntropyLoss(ignore_index=PAD, reduction='sum')
lang_criterion = nn.CrossEntropyLoss(reduction='sum')
print('Parameters:', sum(p.numel() for p in model.parameters()))

Parameters: 812089


## 5. 통합 학습 및 저장

- 처음 실행할 때 `TRAIN=True`로 두세요.
- 번역 손실과 언어 분류 손실을 더해 공유 Encoder를 학습합니다.
- 검증 손실이 최소인 모델과 같은 어휘 사전을 저장합니다.
- 아래의 번역 손실은 패딩이 아닌 정답 토큰당 평균, 언어 손실은 문장당 평균입니다.

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_translation_loss, total_lang_loss = 0.0, 0.0
    total_tokens, total_examples, lang_correct = 0, 0, 0

    for x, y, lang in tqdm(loader, leave=False):
        x, y, lang = x.to(DEVICE), y.to(DEVICE), lang.to(DEVICE)
        with torch.set_grad_enabled(training):
            translation_logits, lang_logits = model(x, y)
            target = y[:, 1:]
            n_tokens = (target != PAD).sum().clamp_min(1)
            translation_loss = translation_criterion(
                translation_logits.reshape(-1, len(vocab)), target.reshape(-1)
            ) / n_tokens
            language_loss = lang_criterion(lang_logits, lang) / lang.numel()
            loss = translation_loss + LANG_LOSS_WEIGHT * language_loss
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        count = int(n_tokens.item())
        total_translation_loss += translation_loss.item() * count
        total_lang_loss += language_loss.item() * lang.numel()
        total_tokens += count
        total_examples += lang.numel()
        lang_correct += int((lang_logits.argmax(-1) == lang).sum().item())

    mean_translation = total_translation_loss / max(total_tokens, 1)
    mean_language = total_lang_loss / max(total_examples, 1)
    return (mean_translation + LANG_LOSS_WEIGHT * mean_language,
            mean_translation, mean_language, lang_correct / max(total_examples, 1))

if TRAIN:
    best_loss = float('inf')
    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        train_stats = run_epoch(train_loader, True)
        val_stats = run_epoch(val_loader, False)
        print(f'Epoch {epoch:02d} | train {train_stats[0]:.4f} | '
              f'val {val_stats[0]:.4f} | '
              f'val_lang_acc {val_stats[3]:.2%} | {time.time()-start:.0f}s')
        if val_stats[0] < best_loss:
            best_loss = val_stats[0]
            torch.save({
                'model': model.state_dict(), 'vocab': vocab,
                'emb_dim': EMB_DIM, 'hidden_dim': HIDDEN_DIM,
                'language_loss_weight': LANG_LOSS_WEIGHT,
            }, MODEL_FILE)
    print('Best validation loss:', round(best_loss, 4))
else:
    print('학습 생략: 저장 모델을 불러옵니다.')

  0%|          | 0/2658 [00:00<?, ?it/s]

  0%|          | 0/335 [00:00<?, ?it/s]

Epoch 01 | train 0.3047 | val 0.0663 | val_lang_acc 100.00% | 77s


  0%|          | 0/2658 [00:00<?, ?it/s]

  0%|          | 0/335 [00:00<?, ?it/s]

Epoch 02 | train 0.0402 | val 0.0249 | val_lang_acc 100.00% | 97s


... (80 outputs left out)

Epoch 29 | train 0.0007 | val 0.0013 | val_lang_acc 100.00% | 91s


  0%|          | 0/2658 [00:00<?, ?it/s]

  0%|          | 0/335 [00:00<?, ?it/s]

Epoch 30 | train 0.0006 | val 0.0010 | val_lang_acc 100.00% | 87s
Best validation loss: 0.0008


## 6. Greedy Decoding: 입력 언어 자동 예측 → 나머지 두 언어 번역

In [ ]:
if not MODEL_FILE.exists():
    raise FileNotFoundError(f'{MODEL_FILE}이 없습니다. TRAIN=True로 먼저 학습하세요.')
checkpoint = torch.load(MODEL_FILE, map_location=DEVICE, weights_only=True)
if checkpoint['vocab'] != vocab:
    raise ValueError('저장된 모델의 어휘가 현재 데이터에서 만든 어휘와 다릅니다.')
if (checkpoint['emb_dim'], checkpoint['hidden_dim']) != (EMB_DIM, HIDDEN_DIM):
    raise ValueError('저장된 모델의 차원 설정이 현재 코드와 다릅니다.')
model.load_state_dict(checkpoint['model'])
model.eval()
print('모델 로드 완료:', MODEL_FILE)

@torch.no_grad()
def translate_all(text, net=model, max_len=80):
    net.eval()
    text = normalize_text(text)
    if not text:
        raise ValueError('입력이 비어 있습니다.')

    missing = sorted(set(ch for ch in tokenize(text) if ch not in stoi))
    if missing:
        print(f'주의: 미등록 문자 {missing} → <unk> 처리')
    x = torch.tensor([ids(text) + [EOS]], dtype=torch.long, device=DEVICE)
    state, lang_logits = net.encode(x)
    source_lang = LANGS[int(lang_logits.argmax(-1).item())]

    results = {}
    for target_lang in LANGS:
        if target_lang == source_lang:
            continue
        # 디코더 시작 토큰만 목표 언어별로 다름
        token = torch.tensor([[stoi[f'<2{target_lang}>']]], device=DEVICE)
        decoder_state = state
        output = []
        finished = False
        for _ in range(max_len):
            hidden, decoder_state = net.decoder(net.embedding(token), decoder_state)
            scores = net.fc(hidden)[:, -1, :]  # 다음 토큰의 점수
            # 특별 토큰이 번역문에 출력되지 않게 함
            scores[:, PAD] = -float('inf')
            scores[:, BOS] = -float('inf')
            scores[:, UNK] = -float('inf')
            for lang in LANGS:
                scores[:, stoi[f'<2{lang}>']] = -float('inf')
            token = scores.argmax(-1).view(1, 1)  # 점수가 가장 높은 토큰 하나만 선택 = Greedy
            token_id = int(token.item())
            if token_id == EOS:
                finished = True
                break
            output.append(itos[token_id])
        results[target_lang] = (''.join(output), finished)
    return source_lang, results

def translate(text, target_lang):
    """자동 언어 판별 후 지정한 하나의 목표 언어로 번역."""
    src, outputs = translate_all(text)
    if target_lang == src:
        raise ValueError(f'입력 언어가 이미 {src}입니다.')
    return outputs[target_lang][0]

모델 로드 완료: lstm_greedy_multitask.pt


## 7. 테스트셋 평가

- 언어 판별의 정확도와 전체 파이프라인의 번역 정확도를 분리해서 확인합니다.
- 정답 기준: (1) 입력 언어 인식 (2) 번역 문자열이 정답과 정확히 일치 (3) EOS 종료에 성공
- `RUN_FULL_TEST`로 전체 평가 실행 여부를 선택할 수 있습니다.

In [7]:
# 먼저 방향별 예제 하나씩 확인
for src in LANGS:
    for tgt in LANGS:
        if src == tgt:
            continue
        sample = next((p for p in test_pairs if p[0] == src and p[1] == tgt), None)
        if sample is None:
            print(f'{src}->{tgt} | 테스트 데이터 없음')
            continue
        _, _, text, gt = sample
        predicted_src, predictions = translate_all(text)
        result, eos_ok = predictions.get(tgt, ('', False))
        ok = predicted_src == src and eos_ok and result == gt
        print(f'{src}->{tgt} | {"PASS" if ok else "FAIL"}')
        if not ok:
            print('  input:', text)
            print('  source predicted:', predicted_src)
            print('  predicted:', result)
            print('  EOS:', eos_ok)
            print('  gt:', gt)

RUN_FULL_TEST = False  # 전체 평가를 원하면 True
if RUN_FULL_TEST:
    counts = {(a, b): [0, 0] for a in LANGS for b in LANGS if a != b}
    lang_correct = lang_total = 0
    # 같은 입력이 두 target에 나타날 수 있으므로 분류 정확도는 레코드가 아니라 번역쌍 기준
    for src, tgt, text, gt in tqdm(test_pairs):
        predicted_src, predictions = translate_all(text)
        correct_source = predicted_src == src
        lang_correct += int(correct_source)
        lang_total += 1
        result, eos_ok = predictions.get(tgt, ('', False))
        correct = correct_source and eos_ok and result == gt
        counts[(src, tgt)][0] += int(correct)
        counts[(src, tgt)][1] += 1
    print(f'Input language accuracy (pair-weighted): {lang_correct / max(lang_total,1):.2%}')
    for (src, tgt), (correct, total) in counts.items():
        print(f'{src}->{tgt}: {correct}/{total} = {correct/max(total,1):.2%}' if total else f'{src}->{tgt}: 데이터 없음')

num->en | PASS
num->ko | PASS
en->num | PASS
en->ko | PASS
ko->num | PASS
ko->en | PASS


## 8. 직접 입력 테스트

In [8]:
def fmt_num(s):
    digits = s.replace(',', '').replace(' ', '')
    if digits.isascii() and digits.isdecimal():
        return re.sub(r'\B(?=(\d{3})+(?!\d))', ',', digits)
    return s

def show(text, lang):
    return fmt_num(text) if lang == 'num' else text

def demo_translate(text):
    source, outputs = translate_all(text)
    print(f'\n모델이 인식한 입력 언어: {LANGS_LABEL[source]} ({source})')
    print(f'입력: {show(normalize_text(text), source)}')
    for tgt, (result, finished) in outputs.items():
        note = '' if finished else ' [주의: EOS 없이 최대 길이 도달]'
        print(f'출력 ({LANGS_LABEL[tgt]}): {show(result, tgt)}{note}')
    print('-' * 45)

while True:
    text = input('\n입력 (종료: q): ').strip()
    if text.lower() == 'q':
        print('테스트를 종료합니다.')
        break
    if not text:
        continue
    try:
        demo_translate(text)
    except (ValueError, RuntimeError) as error:
        print(f'변환 오류: {error}')


모델이 인식한 입력 언어: 숫자 (num)
입력: 12,345
출력 (영어): twelve thousand three hundred forty five
출력 (한국어): 만 이천삼백사십오
---------------------------------------------

모델이 인식한 입력 언어: 영어 (en)
입력: twelve thousand three hundred forty five
출력 (숫자): 12,345
출력 (한국어): 만 이천삼백사십오
---------------------------------------------

모델이 인식한 입력 언어: 한국어 (ko)
입력: 만 이천삼백사십오
출력 (숫자): 12,345
출력 (영어): twelve thousand three hundred forty five
---------------------------------------------
테스트를 종료합니다.


# 9. 더 좋게 만들 수 있을까?
## (1) Pivot Translation
*참고: https://aclanthology.org/Q17-1024.pdf

In [12]:

def pivot_translate(text):
    source, outputs = translate_all(text)

    # 숫자가 입력이면 기존 방식 그대로 번역
    if source == 'num':
        return source, {
            lang: result for lang, (result, finished) in outputs.items()
        }

    # 1단계: 영어 또는 한국어 → 숫자
    number, finished = outputs['num']

    if not finished or not number.isascii() or not number.isdecimal():
        raise ValueError(f'중간 숫자 변환 실패: {number}')

    # 2단계: 숫자 → 반대 언어
    target = 'ko' if source == 'en' else 'en'

    detected, number_outputs = translate_all(number)

    if detected != 'num':
        raise ValueError('중간 숫자를 num으로 인식하지 못했습니다.')

    translated, finished = number_outputs[target]

    if not finished:
        raise ValueError('최종 번역이 정상 종료되지 않았습니다.')

    return source, {
        'num': number,
        target: translated
    }


# 직접 번역 vs Pivot Translation
text = 'thirty thousand'

source, direct = translate_all(text)
_, pivot = pivot_translate(text)

target = 'ko' if source == 'en' else 'en'

print('입력:', text)
print('직접 번역:', direct[target][0])
print('중간 숫자:', pivot['num'])
print('Pivot 번역:', pivot[target])

입력: thirty thousand
직접 번역: 삼만 육
중간 숫자: 30000
Pivot 번역: 삼만 


## (2) Beam as Decoding Algorithm: topk

In [10]:

import torch.nn.functional as F

@torch.no_grad()
def translate_all_beam(text, beam_width=3, max_len=80):
    model.eval()
    text = normalize_text(text)

    if not text:
        raise ValueError('입력이 비어 있습니다.')

    x = torch.tensor(
        [ids(text) + [EOS]],
        dtype=torch.long,
        device=DEVICE
    )

    # Encoder 실행 및 입력 언어 인식
    state, lang_logits = model.encode(x)
    source = LANGS[int(lang_logits.argmax(-1).item())]

    results = {}

    for target in LANGS:
        if target == source:
            continue

        start_token = stoi[f'<2{target}>']

        # (토큰 목록, 누적 로그확률, LSTM 상태, 종료 여부)
        beams = [([start_token], 0.0, state, False)]

        for _ in range(max_len):
            candidates = []

            for tokens, score, hidden_state, finished in beams:
                if finished:
                    candidates.append(
                        (tokens, score, hidden_state, True)
                    )
                    continue

                token = torch.tensor(
                    [[tokens[-1]]],
                    dtype=torch.long,
                    device=DEVICE
                )

                out, new_state = model.decoder(
                    model.embedding(token), hidden_state
                )

                logits = model.fc(out)[:, -1, :].clone()

                # 번역문에 나오면 안 되는 토큰 제외
                logits[:, PAD] = -float('inf')
                logits[:, BOS] = -float('inf')
                logits[:, UNK] = -float('inf')

                for lang in LANGS:
                    logits[:, stoi[f'<2{lang}>']] = -float('inf')

                log_probs = F.log_softmax(logits, dim=-1)

                values, indices = torch.topk(
                    log_probs, beam_width, dim=-1
                )

                for log_prob, token_id in zip(
                    values[0].tolist(), indices[0].tolist()
                ):
                    candidates.append((
                        tokens + [token_id],
                        score + log_prob,
                        new_state,
                        token_id == EOS
                    ))

            # 누적 로그확률이 높은 후보 beam_width개 유지
            beams = sorted(
                candidates,
                key=lambda b: b[1],
                reverse=True
            )[:beam_width]

            if all(b[3] for b in beams):
                break

        # 종료된 후보 중 가장 점수가 높은 경로 선택
        completed = [b for b in beams if b[3]]
        best = max(
            completed if completed else beams,
            key=lambda b: b[1]
        )

        tokens, score, _, finished = best

        # 시작 토큰 및 종료 토큰 제거
        output_ids = tokens[1:]
        if output_ids and output_ids[-1] == EOS:
            output_ids = output_ids[:-1]

        result = ''.join(itos[i] for i in output_ids)
        results[target] = (result, finished)

    return source, results


# Greedy vs Beam Search 비교
text = 'thirty thousand'

source, greedy = translate_all(text)
_, beam = translate_all_beam(text, beam_width=3)

for target in LANGS:
    if target == source:
        continue

    print(f'\n{source} → {target}')
    print('Greedy:', greedy[target][0])
    print('Beam  :', beam[target][0])


en → num
Greedy: 30000
Beam  : 30000

en → ko
Greedy: 삼만 육
Beam  : 삼만 육
